# 02 - Classical ML Baselines and Model Comparison

Цель ноутбука: обучить несколько классических sklearn-моделей, сравнить их с baseline и выбрать лучшую модель по F1 с проверкой ROC-AUC и Recall.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(project_root))

import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.models.train import run_experiment

sns.set_theme(style='whitegrid')

## 1. Запуск эксперимента

In [ ]:
results = run_experiment(save_artifacts=True)
cv_results = results['cv_results']
test_results = results['test_results']
print('Best model:', results['best_model'])

## 2. Cross-validation

In [ ]:
cv_cols = [
    'model', 'cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean',
    'cv_f1_mean', 'cv_roc_auc_mean', 'train_f1_mean', 'train_roc_auc_mean'
]
display(cv_results[cv_cols].round(4))

## 3. Test set

In [ ]:
test_cols = ['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'tn', 'fp', 'fn', 'tp']
display(test_results[test_cols].round(4))

## 4. Сравнение F1 и ROC-AUC

In [ ]:
plot_df = test_results.sort_values('f1')
plt.figure(figsize=(9, 5))
plt.barh(plot_df['model'], plot_df['f1'], label='F1')
plt.scatter(plot_df['roc_auc'], plot_df['model'], color='red', label='ROC-AUC')
plt.xlabel('Метрика')
plt.title('Сравнение моделей на test set')
plt.legend()
plt.show()

## 5. Confusion matrices

In [ ]:
with open(project_root / 'artifacts' / 'confusion_matrices.json', encoding='utf-8') as file:
    matrices = json.load(file)
matrices

## 6. Интерпретируемость

In [ ]:
feature_importance = pd.read_csv(project_root / 'artifacts' / 'feature_importance_top10.csv')
logistic_coefficients = pd.read_csv(project_root / 'artifacts' / 'logistic_coefficients_top.csv')

display(feature_importance.round(4))
display(logistic_coefficients.round(4))

## Вывод

Baseline `DummyClassifier` показывает высокий accuracy только из-за дисбаланса, но F1 равен 0. Лучшая модель по test F1 и ROC-AUC — `GradientBoostingClassifier`. Наиболее важные факторы: жалобы, статус клиента, длительность и интенсивность использования связи.